<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_DRC_Sec_CTP_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Set display options for better readability
pd.set_option('display.float_format', '{:,.2f}'.format)

# ---
# Step 1: Gross Jump-to-Default (JTD) Amounts (Article 325ac(2))
# ---
print("Step 1: Initial Portfolio Data (Gross JTD)")
print("Per Art. 325ac(2), Gross JTD = Market Value.")
data = {
    'Bucket': ['Bucket 1', 'Bucket 1', 'Bucket 2', 'Bucket 2'],
    'Issuer': ['Issuer 1', 'Issuer 1', 'Issuer 2', 'Issuer 2'],
    'Credit_Quality': ['A', 'A', 'BB', 'BB'],
    'Gross_JTD': [4000, -1000, -1900, 800]
}
df = pd.DataFrame(data)
print(df)
print("\n" + "="*60 + "\n")

# ---
# Step 2: Net Jump-to-Default (JTD) Amounts (Article 325ac(4))
# ---
print("Step 2: Calculate Net JTD")
print("Netting identical positions (by Issuer) to find Net JTD.")

# Group by issuer (and other identifiers) and sum to get the Net JTD
net_jtd_df = df.groupby(['Bucket', 'Issuer', 'Credit_Quality'], as_index=False).agg(
    Net_JTD=('Gross_JTD', 'sum')
)
print(net_jtd_df)
print("\n" + "="*60 + "\n")

# ---
# Step 3: Determine Default Risk Weights (RW) (Article 325ad(1)(a))
# ---
print("Step 3: Determine Default Risk Weights (RW)")
print("Assigning RW based on Credit Quality per Article 325y, Table 2.")

# Define the Risk Weight map
rw_map = {
    'AAA': 0.005, 'AA': 0.02, 'A': 0.03, 'BBB': 0.06,
    'BB': 0.15, 'B': 0.30, 'CCC': 0.50, 'Unrated': 0.15, 'Defaulted': 1.00
}

# Apply the map to our net positions
net_jtd_df['RW'] = net_jtd_df['Credit_Quality'].map(rw_map)
print(net_jtd_df[['Bucket', 'Issuer', 'Credit_Quality', 'Net_JTD', 'RW']])
print("\n" + "="*60 + "\n")

# ---
# Step 4: Calculate WtS_ACTP (Hedging Discount Factor) (Article 325ad(3))
# ---
print("Step 4: Calculate WtS_ACTP (Hedging Discount Factor)")
print("Calculating the single, portfolio-wide WtS ratio...")

# Calculate total longs and shorts across the *entire* portfolio
total_longs = net_jtd_df[net_jtd_df['Net_JTD'] > 0]['Net_JTD'].sum()
total_shorts = net_jtd_df[net_jtd_df['Net_JTD'] < 0]['Net_JTD'].abs().sum()
denominator = total_longs + total_shorts

# Avoid division by zero
wts_actp = 0.0 if denominator == 0 else total_longs / denominator

print(f"Total Net Long JTD:     {total_longs:,.2f}")
print(f"Total Absolute Net Short JTD: {total_shorts:,.2f}")
print(f"Denominator (Longs + Shorts): {denominator:,.2f}")
print(f"WtS_ACTP Ratio (Longs / Denominator): {wts_actp:.4f}")
print("\n" + "="*60 + "\n")

# ---
# Step 5: Calculate Bucket-Level DRC (DRC_b) (Article 325ad(3))
# ---
print("Step 5: Calculate Bucket-Level DRC (DRC_b)")
print(f"Applying WtS_ACTP ratio ({wts_actp:.4f}) to each bucket and flooring at 0.")

# First, calculate weighted JTD for all positions
net_jtd_df['Weighted_JTD'] = net_jtd_df['Net_JTD'] * net_jtd_df['RW']

# Now, group by bucket and sum the weighted longs and weighted shorts
bucket_drc_df = net_jtd_df.groupby('Bucket').agg(
    Total_Weighted_Longs=('Weighted_JTD', lambda x: x[x > 0].sum()),
    Total_Weighted_Shorts=('Weighted_JTD', lambda x: x[x < 0].abs().sum())
).reset_index()

# Apply the WtS ratio and the DRC_b formula
bucket_drc_df['Discounted_Shorts'] = wts_actp * bucket_drc_df['Total_Weighted_Shorts']
bucket_drc_df['Pre_Floor_DRC'] = bucket_drc_df['Total_Weighted_Longs'] - bucket_drc_df['Discounted_Shorts']
bucket_drc_df['DRC_b'] = bucket_drc_df['Pre_Floor_DRC'].apply(lambda x: max(x, 0))

print(bucket_drc_df)
print("\n" + "="*60 + "\n")

# ---
# Step 6: Final DRC Aggregation (DRC_ACTP) (Article 325ad(4))
# ---
print("Step 6: Final DRC Aggregation (DRC_ACTP)")
print("Applying the final aggregation formula.")

# Get the DRC_b values from our Step 5 table
drc_b_values = bucket_drc_df['DRC_b']

# Apply the regulatory formula: max{ sum( max(DRC_b, 0) + 0.5 * min(DRC_b, 0) ) ; 0 }
# Since DRC_b is already floored at 0, this simplifies to sum(DRC_b).
processed_drc_b_sum = drc_b_values.apply(lambda drc_b: max(drc_b, 0) + 0.5 * min(drc_b, 0)).sum()
final_drc_actp = max(processed_drc_b_sum, 0)

print(f"Bucket 1 DRC_b: {bucket_drc_df[bucket_drc_df['Bucket']=='Bucket 1']['DRC_b'].values[0]:,.2f}")
print(f"Bucket 2 DRC_b: {bucket_drc_df[bucket_drc_df['Bucket']=='Bucket 2']['DRC_b'].values[0]:,.2f}")
print(f"\nSum of processed DRC_b values: {processed_drc_b_sum:,.2f}")
print(f"\nFinal DRC_ACTP = max( {processed_drc_b_sum:,.2f} ; 0 )")
print(f"\nFinal DRC_ACTP = {final_drc_actp:,.2f}")

Step 1: Initial Portfolio Data (Gross JTD)
Per Art. 325ac(2), Gross JTD = Market Value.
     Bucket    Issuer Credit_Quality  Gross_JTD
0  Bucket 1  Issuer 1              A       4000
1  Bucket 1  Issuer 1              A      -1000
2  Bucket 2  Issuer 2             BB      -1900
3  Bucket 2  Issuer 2             BB        800


Step 2: Calculate Net JTD
Netting identical positions (by Issuer) to find Net JTD.
     Bucket    Issuer Credit_Quality  Net_JTD
0  Bucket 1  Issuer 1              A     3000
1  Bucket 2  Issuer 2             BB    -1100


Step 3: Determine Default Risk Weights (RW)
Assigning RW based on Credit Quality per Article 325y, Table 2.
     Bucket    Issuer Credit_Quality  Net_JTD   RW
0  Bucket 1  Issuer 1              A     3000 0.03
1  Bucket 2  Issuer 2             BB    -1100 0.15


Step 4: Calculate WtS_ACTP (Hedging Discount Factor)
Calculating the single, portfolio-wide WtS ratio...
Total Net Long JTD:     3,000.00
Total Absolute Net Short JTD: 1,100.00
Denomin